# SMILES Standardization and Descriptor Calculation Tutorial

This notebook demonstrates how to:
1. Standardize SMILES strings using RDKit
2. Calculate molecular descriptors using RDKit and Mordred
3. Generate polymer-specific descriptors

Perfect for beginners to cheminformatics and polymer property prediction!

## Section 1: Import Required Libraries

Let's start by importing all the necessary libraries for cheminformatics work.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# RDKit for cheminformatics
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import rdmolops

# Mordred for comprehensive descriptors
from mordred import Calculator, descriptors

# Import our custom data_prep module
import sys
sys.path.append('../src')
from data_prep import SMILESProcessor, DescriptorCalculator

print("✅ All libraries imported successfully!")
print(f"RDKit version: {Chem.__version__}")

## Section 2: Load Sample SMILES Data

Let's create some example polymer SMILES strings to work with. These represent common polymer repeat units.

In [ ]:
# Create sample polymer dataset
sample_polymers = {
    'Polymer_Name': [
        'Polystyrene repeat unit',
        'Polyethylene repeat unit', 
        'Polypropylene repeat unit',
        'PET repeat unit',
        'PMMA repeat unit',
        'With salt (needs cleaning)',
        'Invalid SMILES (will fail)'
    ],
    'SMILES': [
        'C(c1ccccc1)C',  # Polystyrene
        'CC',  # Polyethylene
        'CC(C)C',  # Polypropylene  
        'O=C(c1ccc(cc1)C(=O)O)O',  # PET aromatic part
        'COC(=O)C(C)(C)C',  # PMMA
        'CCO.Cl',  # Ethanol with chloride salt
        'invalid_smiles_string'  # Invalid
    ],
    'Tg_experimental': [373, 200, 250, 342, 378, 300, np.nan]  # Glass transition temps in K
}

df = pd.DataFrame(sample_polymers)
print("📊 Sample polymer dataset:")
print(df)
print(f"\nDataset shape: {df.shape}")

## Section 3: SMILES Standardization with RDKit

Let's see how RDKit canonicalizes SMILES strings. Canonicalization ensures that different representations of the same molecule are converted to a standard form.

In [ ]:
# Initialize our SMILES processor
processor = SMILESProcessor()

# Demonstrate canonicalization
print("🔄 SMILES Canonicalization Examples:")
print("=" * 50)

test_smiles = [
    'C(c1ccccc1)C',  # Original
    'CC(c1ccccc1)',  # Different representation
    'c1ccccc1CC',    # Another representation
]

for smiles in test_smiles:
    canonical = processor.canonicalize_smiles(smiles)
    print(f"Original:   {smiles}")
    print(f"Canonical:  {canonical}")
    print("-" * 30)

# Apply to our dataset
print("\n📋 Applying canonicalization to dataset:")
df['SMILES_canonical'] = df['SMILES'].apply(processor.canonicalize_smiles)

# Show results
comparison_df = df[['Polymer_Name', 'SMILES', 'SMILES_canonical']].copy()
print(comparison_df)

## Section 4: Handle Stereochemistry

RDKit can handle stereochemical information in SMILES. Let's see how it works with chiral centers and double bond geometry.

In [ ]:
# Examples with stereochemistry
stereo_examples = [
    'C[C@H](O)CC',    # R stereocenter 
    'C[C@@H](O)CC',   # S stereocenter
    'C/C=C/C',        # E double bond
    'C/C=C\\C',       # Z double bond
]

print("🧬 Stereochemistry Examples:")
print("=" * 40)

for smiles in stereo_examples:
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        # Assign stereochemistry explicitly
        Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
        canonical = Chem.MolToSmiles(mol, isomericSmiles=True)
        print(f"Original:      {smiles}")
        print(f"With stereo:   {canonical}")
        print("-" * 30)
    else:
        print(f"Invalid SMILES: {smiles}")

# For polymers, stereochemistry is important for tacticity
print("\n🔗 Polymer Tacticity Examples:")
tacticity_examples = [
    ('[*]C([*])C', 'Atactic (no specific stereochemistry)'),
    ('[*][C@H]([*])C', 'Isotactic (all same chirality)'),
    ('[*][C@H]([*])C.[*][C@@H]([*])C', 'Syndiotactic (alternating)')
]

for smiles, description in tacticity_examples:
    print(f"{description}: {smiles}")

## Section 5: Remove Salts and Fragments

RDKit can automatically remove salts and select the largest fragment from SMILES containing multiple components.

In [ ]:
# Examples with salts and fragments
salt_examples = [
    'CCO.Cl',           # Ethanol with chloride
    'CC(=O)O.[Na+]',    # Sodium acetate
    'CC.O.CC',          # Multiple fragments
    'c1ccccc1.CCO',     # Benzene + ethanol
]

print("🧂 Salt Removal Examples:")
print("=" * 35)

for smiles in salt_examples:
    cleaned = processor.remove_salts(smiles)
    print(f"Original:  {smiles}")
    print(f"Cleaned:   {cleaned}")
    print("-" * 25)

# Apply full standardization to our dataset
print("\n🔧 Full Standardization Pipeline:")
df['SMILES_standardized'] = df['SMILES'].apply(processor.standardize_smiles)

# Show before and after
standardization_df = df[['Polymer_Name', 'SMILES', 'SMILES_standardized']].copy()
print(standardization_df)

# Count valid vs invalid
valid_count = df['SMILES_standardized'].notna().sum()
total_count = len(df)
print(f"\n✅ Successfully standardized: {valid_count}/{total_count} SMILES")

## Section 6: Generate RDKit Descriptors

Now let's calculate basic molecular descriptors using RDKit. These are fundamental properties like molecular weight, lipophilicity, etc.

In [ ]:
# Initialize descriptor calculator
desc_calc = DescriptorCalculator()

# Filter to valid SMILES only
valid_df = df[df['SMILES_standardized'].notna()].copy()

print("📊 Calculating RDKit Descriptors...")
print("=" * 35)

# Calculate descriptors for each valid SMILES
rdkit_descriptors = []
for idx, row in valid_df.iterrows():
    smiles = row['SMILES_standardized']
    desc = desc_calc.calculate_rdkit_descriptors(smiles)
    desc['Polymer_Name'] = row['Polymer_Name']
    rdkit_descriptors.append(desc)

rdkit_df = pd.DataFrame(rdkit_descriptors)

# Display results
descriptor_cols = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'NumRings', 'NumRotatableBonds']
display_df = rdkit_df[['Polymer_Name'] + descriptor_cols]

print("🎯 RDKit Descriptors Results:")
print(display_df.round(2))

# Explain what each descriptor means
print("\n📚 Descriptor Explanations:")
explanations = {
    'MW': 'Molecular Weight (g/mol)',
    'LogP': 'Lipophilicity (partition coefficient)',
    'HBD': 'Hydrogen Bond Donors count',
    'HBA': 'Hydrogen Bond Acceptors count', 
    'TPSA': 'Topological Polar Surface Area (Ų)',
    'NumRings': 'Number of rings',
    'NumRotatableBonds': 'Number of rotatable bonds (flexibility)'
}

for abbrev, explanation in explanations.items():
    print(f"  {abbrev}: {explanation}")